In [2]:
import json
from pathlib import Path
import pandas as pd

# Load all cleaned json files from data/interim and merge them
interim_dir = Path("../data/interim")

records = []
for path in sorted(interim_dir.glob("cleaned_*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    print(f"{path.name}: {len(data)} records")
    records.extend(data)

print(f"\nTotal: {len(records)} episodes")

cleaned_series_1.json: 419 records
cleaned_series_10.json: 3 records
cleaned_series_100.json: 188 records
cleaned_series_1000.json: 1 records
cleaned_series_101.json: 15 records
cleaned_series_1017.json: 5 records
cleaned_series_1019.json: 10 records
cleaned_series_102.json: 24 records
cleaned_series_1020.json: 8 records
cleaned_series_1023.json: 2 records
cleaned_series_1024.json: 8 records
cleaned_series_1025.json: 2 records
cleaned_series_1026.json: 7 records
cleaned_series_1027.json: 5 records
cleaned_series_1028.json: 3 records
cleaned_series_103.json: 3 records
cleaned_series_1030.json: 5 records
cleaned_series_1031.json: 30 records
cleaned_series_1032.json: 2 records
cleaned_series_1033.json: 13 records
cleaned_series_1034.json: 1 records
cleaned_series_1035.json: 1 records
cleaned_series_1036.json: 77 records
cleaned_series_1037.json: 2 records
cleaned_series_1038.json: 4 records
cleaned_series_1039.json: 2 records
cleaned_series_104.json: 8 records
cleaned_series_1041.json: 1 

In [3]:
df = pd.DataFrame([
    {
        "title":            r.get("title"),
        "series_name":      r.get("series_name"),
        "episode_number":   r.get("episode_number"),
        "description":      r.get("description"),
        "duration_minutes": r.get("duration_minutes"),
        "release_date":     r.get("release_date"),
        "label":            r.get("label"),
        "cover_url":        r.get("cover_url"),
        "order_number":     r.get("order_number"),
        "source_url":       r.get("source_url"),
        "n_speakers":       len(r.get("speakers") or []),
        "n_genres":         len(r.get("genres") or []),
    }
    for r in records
])

print(f"Shape: {df.shape}")

Shape: (14321, 12)


In [4]:
# Data types and memory usage
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14321 entries, 0 to 14320
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             14321 non-null  str    
 1   series_name       14321 non-null  str    
 2   episode_number    14321 non-null  int64  
 3   description       14311 non-null  str    
 4   duration_minutes  6224 non-null   float64
 5   release_date      10917 non-null  str    
 6   label             14321 non-null  str    
 7   cover_url         11840 non-null  str    
 8   order_number      7622 non-null   str    
 9   source_url        0 non-null      object 
 10  n_speakers        14321 non-null  int64  
 11  n_genres          14321 non-null  int64  
dtypes: float64(1), int64(3), object(1), str(7)
memory usage: 1.3+ MB


In [5]:
# Missing values — how complete is the data?
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing": missing, "percent": missing_pct})

,missing,percent
source_url,14321,100.0
duration_minutes,8097,56.5
order_number,6699,46.8
release_date,3404,23.8
cover_url,2481,17.3
description,10,0.1
title,0,0.0
series_name,0,0.0
episode_number,0,0.0
label,0,0.0


In [6]:
# Episodes without a title — should be 0, otherwise parser issue
no_title = df[df["title"].isnull()]
print(f"Episodes without title: {len(no_title)}")
if len(no_title) > 0:
    print(no_title[["source_url"]].head(10))

Episodes without title: 0


In [7]:
# Series: count and episodes per series
series_counts = df.groupby("series_name").size().sort_values(ascending=False)
print(f"Number of series: {len(series_counts)}\n")
series_counts.head(20)

Number of series: 1279



series_name
(Diverse)                 419
Die Drei ???              337
TKKG                      285
Märchen                   278
John Sinclair             226
Gruselkabinett            211
Benjamin Blümchen         206
Fünf Freunde              188
Bibi Blocksberg           154
Die Drei ??? Kids         126
Karl May                  118
Jan Tenner                117
Midnight Tales            116
Hanni und Nanni           116
Offenbarung 23            111
Teufelskicker             110
Die drei !!!              109
Sherlock Holmes           101
Filme                      99
Europa - Die Originale     97
dtype: int64

In [8]:
# Parse release date and check for problems
df["release_date_parsed"] = pd.to_datetime(
    df["release_date"], format="%d.%m.%Y", errors="coerce"
)

n_missing = df["release_date"].isnull().sum()
n_failed  = df["release_date_parsed"].isnull().sum()
n_errors  = n_failed - n_missing

print(f"No date at all:       {n_missing}")
print(f"Failed to parse:      {n_errors}  ← cleaning candidates")
print(f"Successfully parsed:  {len(df) - n_failed}")

# Which values failed?
df[df["release_date"].notnull() & df["release_date_parsed"].isnull()][
    ["title", "release_date", "source_url"]
].head(20)

No date at all:       3404
Failed to parse:      0  ← cleaning candidates
Successfully parsed:  10917


,title,release_date,source_url


In [17]:
# Releases per year
df["year"] = df["release_date_parsed"].dt.year
df["year"].value_counts().sort_index()

year
1089.0      2
1910.0      1
1912.0      1
1938.0      1
1956.0      1
1960.0      2
1962.0      1
1963.0      1
1969.0      2
1976.0      3
1979.0      9
1980.0     12
1981.0      7
1982.0      2
1983.0      4
1984.0      2
1985.0      2
1986.0      3
1987.0      4
1988.0      8
1989.0      2
1990.0      1
1991.0      6
1992.0     10
1993.0     10
1994.0     13
1995.0     13
1996.0     15
1997.0     27
1998.0     43
1999.0     40
2000.0     71
2001.0    291
2002.0    328
2003.0    321
2004.0    430
2005.0    489
2006.0    683
2007.0    779
2008.0    758
2009.0    692
2010.0    661
2011.0    585
2012.0    408
2013.0    351
2014.0    255
2015.0    299
2016.0    282
2017.0    284
2018.0    263
2019.0    229
2020.0    320
2021.0    443
2022.0    328
2023.0    360
2024.0    318
2025.0    314
2026.0    125
2027.0      1
2028.0      1
Name: count, dtype: int64

In [10]:
# Outliers in duration
print(df["duration_minutes"].describe())

print("\nVery short (< 5 min):")
print(df[df["duration_minutes"] < 5][["title", "series_name", "duration_minutes"]].head(10))

print("\nVery long (> 180 min):")
print(df[df["duration_minutes"] > 180][["title", "series_name", "duration_minutes"]].head(10))

count    6224.000000
mean       58.494762
std        62.338367
min         2.100000
25%        47.082500
50%        58.000000
75%        68.272500
max      4720.000000
Name: duration_minutes, dtype: float64

Very short (< 5 min):
                                    title                     series_name  \
1319                          6. Dezember  Die Drei ??? und der 5. Advent   
1327                         14. Dezember  Die Drei ??? und der 5. Advent   
1329                         16. Dezember  Die Drei ??? und der 5. Advent   
1331                         18. Dezember  Die Drei ??? und der 5. Advent   
1333                         20. Dezember  Die Drei ??? und der 5. Advent   
1334                         21. Dezember  Die Drei ??? und der 5. Advent   
4446               Folge 1 - Die Songs EP                     Eazy Eizbär   
4484    Die gefährliche Spur (Käse-Pizza)              Die Drei ??? Pizza   
4485  Der feuerrote Teufel (Salami-Pizza)              Die Drei ??? Pizza   


In [ ]:
# Explore genres and count
genres_series = pd.Series([
    g for r in records for g in (r.get("genres") or [])
])
print(f"Unique genres: {genres_series.nunique()}\n")
genres_series.value_counts().head(20)

Unique genres: 20



Krimi und Detektiv         1241
Grusel und Horror          1108
Abenteuer                  1015
Kinder und Jugend           947
Mystery                     646
Science Fiction             474
Action                      349
Fantasy                     232
Comedy                      173
OTV                         155
Für Mädchen                 124
Wissen und Infotainment     104
Märchen                      92
Liebe und Romantik           91
Western                      80
Thriller                     65
Musik und Lieder             62
Nur ab 18                    42
Pferdegeschichten            40
Für Jungs                    27
Name: count, dtype: int64

In [12]:
# Most active voice actors
all_speakers = pd.Series([
    s["speaker"] for r in records for s in (r.get("speakers") or [])
])
print(f"Total speaker credits: {len(all_speakers)}")
print(f"Unique speakers:       {all_speakers.nunique()}\n")
all_speakers.value_counts().head(20)

Total speaker credits: 99549
Unique speakers:       10527



Rohrbeck, Oliver          602
Mackensy, Lutz            571
Fröhlich, Andreas         482
von der Meden, Andreas    433
Wawrczeck, Jens           419
Draeger, Sascha           390
Paetsch, Hans             362
Schülke, Achim            354
Halver, Konrad            338
Missler, Robert           333
Schneider, Reinhilt       325
Karallus, Thomas          315
Krauss, Helmut            312
Riedel, Lutz              312
Hagen, Till               312
Thormann, Jürgen          310
Lubowski, Manou           306
Schmidt-Foß, Gerrit       305
Bierstedt, Detlef         303
Bach, Patrick             300
Name: count, dtype: int64